In [1]:
import pandas as pd

# Replace 'your_file.csv' with the actual file name
df = pd.read_csv('D:/sathya/project/malicious_phish.csv')

# Display the first few rows of the dataframe to verify the content
print(df.head())

# Classify unique values in the 'type' column
unique_types = df['type'].unique()

# Create a new DataFrame with classified unique values
classified_df = pd.DataFrame(unique_types, columns=['Unique Types'])

# Display the unique types
print(classified_df)

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
  Unique Types
0     phishing
1       benign
2   defacement
3      malware


#preprocessing

In [2]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('D:/sathya/project/malicious_phish.csv')

# Inspect the first few rows
print(df.head())


                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [3]:
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np
df = pd.read_csv('D:/sathya/project/malicious_phish.csv')

# Tokenize sentences and words
tokenized_docs = []
for doc in df:
    sentences = sent_tokenize(doc)  # Split document into sentences
    tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]  # Split sentences into words
    tokenized_docs.append(tokenized_sentences)


In [4]:
from collections import Counter

# Flatten the list of token lists and create a Counter object
all_tokens = [token for tokenized_docs in df['tokenized_docs'] for token in tokenized_docs]
vocabulary = Counter(all_tokens)

# Optional: Set a maximum vocabulary size to limit the number of tokens
max_vocab_size = 5000
vocab = {token: idx for idx, (token, _) in enumerate(vocabulary.most_common(max_vocab_size), 1)}


KeyError: 'tokenized_docs'

In [ ]:
def tokens_to_indices(tokens, vocab):
    return [vocab.get(token, 0) for token in tokens]  # Use 0 for unknown tokens

df['token_indices'] = df['tokens'].apply(lambda tokens: tokens_to_indices(tokens, vocab))


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define maximum lengths
max_sentence_length = 3  # Maximum number of sentences per document
max_word_length = 5  # Maximum number of words per sentence

# Initialize the tokenizer (for words)
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=5000, oov_token="<OOV>")
all_sentences = [word for doc in tokenized_docs for sentence in doc for word in sentence]
tokenizer.fit_on_texts(all_sentences)

# Convert words to integer indices and pad
processed_docs = []
for doc in tokenized_docs:
    encoded_sentences = [
        tokenizer.texts_to_sequences(sentence) for sentence in doc
    ]  # Convert each word to its index
    padded_sentences = pad_sequences(encoded_sentences, maxlen=max_word_length)  # Pad words
    processed_docs.append(padded_sentences)

# Pad sentences to ensure uniform sentence count
padded_docs = pad_sequences(processed_docs, maxlen=max_sentence_length, padding='post', dtype='object')

# Convert to NumPy array with correct shape
X_hierarchical = np.array([np.vstack(sentences) for sentences in padded_docs])
print(f"Shape of input data: {X_hierarchical.shape}")  # (num_samples, max_sentence_length, max_word_length)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(labels)  # Replace `labels` with your actual label list


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, TimeDistributed, Flatten, Attention
from tensorflow.keras.models import Model

# Word Encoder
word_input = Input(shape=(max_word_length,), dtype='int32', name='word_input')
word_embedding = Embedding(input_dim=max_vocab_size + 1, output_dim=128, input_length=max_word_length)(word_input)
word_lstm = Bidirectional(LSTM(64, return_sequences=True))(word_embedding)
word_attention = Attention(name='word_attention')([word_lstm, word_lstm])  # Self-attention mechanism
word_encoder = Model(inputs=word_input, outputs=word_attention)

# Sentence Encoder
sentence_input = Input(shape=(max_sentence_length, max_word_length), dtype='int32', name='sentence_input')
sentence_encoded = TimeDistributed(word_encoder)(sentence_input)
sentence_lstm = Bidirectional(LSTM(64, return_sequences=True))(sentence_encoded)
sentence_attention = Attention(name='sentence_attention')([sentence_lstm, sentence_lstm])  # Self-attention mechanism

# Output Layer
dense_layer = Dense(64, activation='relu')(sentence_attention)
output_layer = Dense(4, activation='softmax')(dense_layer)  # 4 classes: phishing, benign, defacement, malware

# Compile the Model
model = Model(inputs=sentence_input, outputs=output_layer)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Model Summary
model.summary()


In [ ]:
# Train the model
model.fit(X_hierarchical, y_encoded, epochs=5, batch_size=64, validation_split=0.1)

In [ ]:
#evalaate the model
test_loss, test_accuracy = model.evaluate(X_test_hierarchical, y_test_encoded)
print(f'Test Accuracy: {test_accuracy:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Compute the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)

# Print the confusion matrix
print('Confusion Matrix:')
print(conf_matrix)

# Print the classification report
print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))


In [ ]:
import numpy as np
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print('Confusion Matrix:')
print(conf_matrix)

print('Classification Report:')
print(classification_report(y_test, y_pred_classes, target_names=label_mapping.keys()))

# Calculate accuracy from confusion matrix
correct_predictions = np.trace(conf_matrix)
total_predictions = np.sum(conf_matrix)
accuracy = correct_predictions / total_predictions
print(f'Calculated Accuracy from Confusion Matrix: {accuracy:.4f}')

In [ ]:
pwd